[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C48_Cloud_Deployment_Course/01_containerization/01_containerization.ipynb)

# 01 · 容器化：可复现的运行时（从零模拟镜像分层、层缓存与冷启动账）

目标：把 **分层文件系统 → 层缓存链式失效 → 多阶段瘦身 → 冷启动分解** 从零实现，
每个机制都**对拍**朴素参考、每个决策都**算一笔账**。

路线：内容寻址的层 → overlay 合并 → Dockerfile 解析 → 缓存命中链 → 构建成本模型 → 冷启动分解 → ✏️ 练习 → 📖 答案 → 🧪 权重分发账。

> 心智模型：**镜像 = 一叠内容寻址的只读层；缓存 = 一条会级联失效的链；冷启动 = 五项之和**。
> 本环境没有 Docker，但镜像的全部语义都是可以用 Python 精确复现的。

## 1 · 层：内容寻址与 overlay 合并

一层 = 一份文件系统增量（新增/修改/删除）。层的身份 = 其内容的 sha256。
运行时把层从下往上叠，上层覆盖下层；删除用 **whiteout 标记**表示。

先把这套语义实现出来，并验证内容寻址的核心性质：**同内容 → 同摘要 → 只存一份**。

In [ ]:
import hashlib, json, math, re
from dataclasses import dataclass, field

WHITEOUT = '<<deleted>>'      # 模拟 overlayfs 的 .wh. 白出标记

def layer_digest(files: dict) -> str:
    '''层的内容寻址摘要：对 (路径, 内容) 的规范化序列化取 sha256。'''
    payload = json.dumps(sorted(files.items()), ensure_ascii=False).encode()
    return 'sha256:' + hashlib.sha256(payload).hexdigest()[:12]

def overlay(layers):
    '''从下往上合并层，返回最终可见的文件系统。'''
    fs = {}
    for lyr in layers:
        for path, content in lyr.items():
            if content == WHITEOUT:
                fs.pop(path, None)     # 白出：上层把下层的文件遮掉
            else:
                fs[path] = content
    return fs

L1 = {'/usr/bin/python': 'ELF...', '/etc/os-release': 'debian'}
L2 = {'/usr/lib/git': 'bin', '/var/cache/apt/pkg.deb': 'x' * 100}
L3 = {'/var/cache/apt/pkg.deb': WHITEOUT}      # 「清理」apt 缓存
L4 = {'/app/main.py': 'print(1)'}

fs = overlay([L1, L2, L3, L4])
print('最终可见文件:', sorted(fs))
assert '/var/cache/apt/pkg.deb' not in fs, '被白出的文件不该可见'
assert fs['/app/main.py'] == 'print(1)'
# 内容寻址：同内容必同摘要，不同内容必不同摘要
assert layer_digest(L1) == layer_digest(dict(L1)), '同内容 -> 同摘要（去重的基础）'
assert layer_digest(L1) != layer_digest(L2)
print('✅ overlay 合并与内容寻址正确')

### 「删除不会让镜像变小」——用字节数证明它

这是分层最反直觉的性质。白出标记只影响**可见性**，不影响**镜像体积**。

In [ ]:
def visible_bytes(layers):
    return sum(len(v) for v in overlay(layers).values())

def image_bytes(layers):
    '''镜像实际体积 = 所有层的字节之和（白出标记本身也占一点，忽略）。'''
    return sum(len(v) for lyr in layers for v in lyr.values() if v != WHITEOUT)

vis, img = visible_bytes([L1, L2, L3, L4]), image_bytes([L1, L2, L3, L4])
print(f'可见字节 {vis}  vs  镜像实际字节 {img}')
assert img > vis, '删除文件后，镜像仍然携带那些字节！'

# 正确做法：在同一层内「装了再删」，那些字节根本不会被写进任何层
L23_merged = {'/usr/lib/git': 'bin'}          # RUN apt-get install ... && rm -rf ...
img_good = image_bytes([L1, L23_merged, L4])
print(f'同层内装了再删: 镜像字节 {img_good}（省下 {img - img_good} 字节）')
assert img_good < img
print('✅ 证毕：RUN a && rm b 必须写在同一条指令里，分两条 RUN 等于没删')

## 2 · Dockerfile 解析与层缓存的链式失效

缓存规则只有一条，但它是**链式**的：

> 第 i 层缓存有效 ⟺ 第 i−1 层缓存有效 **且** 第 i 条指令及其输入未变。

一旦某层失效，其后所有层全部失效。下面实现一个最小构建器来复现这条语义。

In [ ]:
@dataclass
class Instr:
    op: str            # FROM / COPY / RUN
    arg: str
    inputs: tuple = () # 该指令依赖的外部文件（COPY 的源）
    cost_s: float = 1.0  # 重建这层要多久
    size_mb: float = 0.0

def instr_key(ins, file_hashes):
    '''指令的缓存 key = 指令本身 + 其外部输入的内容哈希。'''
    ext = tuple(file_hashes.get(f, '') for f in ins.inputs)
    return hashlib.sha256(f'{ins.op}|{ins.arg}|{ext}'.encode()).hexdigest()[:12]

def build(dockerfile, file_hashes, cache):
    '''返回 (总耗时, 每层是否命中, 新缓存)。cache: 上一次构建的层 key 列表。'''
    total, hits, keys, chain_ok = 0.0, [], [], True
    for i, ins in enumerate(dockerfile):
        k = instr_key(ins, file_hashes)
        hit = chain_ok and i < len(cache) and cache[i] == k
        if not hit:
            chain_ok = False          # ← 链式失效：一旦断掉，后面全部重建
            total += ins.cost_s
        hits.append(hit); keys.append(k)
    return total, hits, keys

BAD = [
    Instr('FROM', 'python:3.11-slim', cost_s=5,  size_mb=130),
    Instr('COPY', '. /app', inputs=('src',), cost_s=1, size_mb=12),
    Instr('RUN',  'pip install -r requirements.txt', inputs=(), cost_s=300, size_mb=2100),
]
GOOD = [
    Instr('FROM', 'python:3.11-slim', cost_s=5,  size_mb=130),
    Instr('COPY', 'requirements.txt .', inputs=('req',), cost_s=1, size_mb=1),
    Instr('RUN',  'pip install -r requirements.txt', inputs=(), cost_s=300, size_mb=2100),
    Instr('COPY', '. /app', inputs=('src',), cost_s=1, size_mb=12),
]
print('两份 Dockerfile 已定义：BAD 把 COPY . 放在 pip 之前，GOOD 放在之后')

In [ ]:
# 第一次构建：全部 miss（冷构建）
fh0 = {'src': 'v1', 'req': 'r1'}
t_bad0, _, cache_bad = build(BAD, fh0, [])
t_good0, _, cache_good = build(GOOD, fh0, [])
print(f'冷构建: BAD {t_bad0:.0f}s | GOOD {t_good0:.0f}s  （都要装一次依赖，差不多）')

# 第二次构建：只改了应用代码（日常最高频的场景）
fh1 = {'src': 'v2', 'req': 'r1'}
t_bad1, hits_bad, _  = build(BAD,  fh1, cache_bad)
t_good1, hits_good, _ = build(GOOD, fh1, cache_good)
print(f'\n改一行代码后重建:')
print(f'  BAD  {t_bad1:>6.0f}s  层命中 {hits_bad}')
print(f'  GOOD {t_good1:>6.0f}s  层命中 {hits_good}')

assert t_good1 < t_bad1, 'GOOD 顺序应显著更快'
assert t_bad1 >= 300, 'BAD 顺序会重装依赖'
assert hits_good[:3] == [True, True, True], 'GOOD 的前三层应全部命中'
print(f'\n✅ 同样的内容、只是顺序不同 -> 日常重建快 {t_bad1/t_good1:.0f} 倍')

### 链式失效的可视化：改依赖清单会怎样？

注意 GOOD 顺序并非万能——改 `requirements.txt` 时它同样要重装。
缓存优化的本质是**把高频变更排到链尾**，而不是消除重建。

In [ ]:
# 注意：cache_bad / cache_good 都是相对基线 fh0 = {'src':'v1', 'req':'r1'} 建立的
scenarios = [
    ('只改代码',        {'src': 'v2', 'req': 'r1'}),
    ('只改依赖清单',    {'src': 'v1', 'req': 'r2'}),
    ('都改',            {'src': 'v2', 'req': 'r2'}),
    ('什么都没改',      {'src': 'v1', 'req': 'r1'}),
]
print(f"{'场景':<14s} {'BAD(s)':>8s} {'GOOD(s)':>9s}")
for name, fh in scenarios:
    tb, _, _ = build(BAD,  fh, cache_bad)
    tg, _, _ = build(GOOD, fh, cache_good)
    print(f'{name:<14s} {tb:>8.0f} {tg:>9.0f}')

# 「什么都没改」两者都应全命中、0 秒
assert build(BAD,  fh0, cache_bad)[0]  == 0
assert build(GOOD, fh0, cache_good)[0] == 0
# 「只改依赖清单」时 GOOD 也躲不掉重装 —— 缓存优化不是消除重建，是把高频变更排到链尾
assert build(GOOD, {'src':'v1','req':'r2'}, cache_good)[0] >= 300
print('\n✅ 完全未变时两者都 0 秒（全命中）；差距只体现在**高频**变更上')

## 3 · 期望构建成本：为什么「少变的放前面」是可证明的

把顺序问题形式化：设第 i 条指令日变更概率 `p_i`、重建成本 `c_i`。
因为失效级联，第 i 层变更会导致 `Σ_{j≥i} c_j` 的重建。日期望成本：

$$E = \sum_i p_i \cdot \sum_{j \ge i} c_j$$

这个和式解释了全部直觉：**高 p 的指令应尽量靠后**，因为靠后的后缀和更小。

In [ ]:
def expected_build_cost(instrs, probs):
    '''E = Σ_i p_i * (后缀成本和)。'''
    n = len(instrs)
    suffix = [0.0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suffix[i] = suffix[i + 1] + instrs[i].cost_s
    return sum(p * suffix[i] for i, p in enumerate(probs))

# 应用代码天天改(p=0.9)，依赖清单偶尔改(p=0.05)，基础镜像极少改(p=0.01)
p_bad  = [0.01, 0.90, 0.05]              # FROM, COPY ., RUN pip
p_good = [0.01, 0.05, 0.05, 0.90]        # FROM, COPY req, RUN pip, COPY .
E_bad  = expected_build_cost(BAD,  p_bad)
E_good = expected_build_cost(GOOD, p_good)
print(f'日期望构建耗时: BAD {E_bad:.1f}s | GOOD {E_good:.1f}s')
assert E_good < E_bad
print(f'✅ 期望成本降低 {(1 - E_good/E_bad)*100:.0f}%，与前面的模拟结论一致')

# 对拍：用蒙特卡洛直接模拟 2000 天，验证解析式
import random
def monte_carlo(instrs, probs, days=2000, seed=0):
    rng = random.Random(seed); total = 0.0
    for _ in range(days):
        first_changed = None
        for i, p in enumerate(probs):
            if rng.random() < p:
                first_changed = i; break
        if first_changed is not None:
            total += sum(ins.cost_s for ins in instrs[first_changed:])
    return total / days

mc_good = monte_carlo(GOOD, p_good)
print(f'蒙特卡洛(GOOD) {mc_good:.1f}s  vs  解析式 {E_good:.1f}s')
assert abs(mc_good - E_good) / E_good < 0.20, '解析式应与模拟在同一量级（解析式忽略了同日多层同变）'
print('✅ 对拍通过：解析式与蒙特卡洛一致')

## 4 · 多阶段构建：把编译期字节挡在最终镜像之外

多阶段 = 在 builder 阶段用 devel 基础镜像编译，只把**产物**拷进 runtime 阶段。
下面量化它省下多少字节、多少拉取时间。

In [ ]:
DEVEL_BASE, RUNTIME_BASE = 6500, 2400     # MB，公开量级 (cuda:12.1 devel vs runtime)
DEPS, TOOLCHAIN, APP, ARTIFACT = 2100, 1800, 12, 900   # MB

single_stage = DEVEL_BASE + DEPS + TOOLCHAIN + APP        # 全塞一个阶段
multi_stage  = RUNTIME_BASE + ARTIFACT + APP              # 只拷产物

def pull_seconds(mb, bandwidth_mbps=500):
    return mb / bandwidth_mbps

print(f'单阶段镜像: {single_stage:>6d} MB  -> 拉取 {pull_seconds(single_stage):>5.1f}s')
print(f'多阶段镜像: {multi_stage:>6d} MB  -> 拉取 {pull_seconds(multi_stage):>5.1f}s')
print(f'节省: {single_stage - multi_stage} MB ({(1-multi_stage/single_stage)*100:.0f}%)')

assert multi_stage < single_stage / 2, '多阶段应至少砍掉一半'
assert pull_seconds(multi_stage) < 10, '多阶段镜像应能在 10 秒内拉完（500MB/s）'
print('✅ 多阶段把冷启动的拉取项从 ~21s 压到 ~7s')

### 层复用：20 个服务共享一个底座能省多少？

内容寻址让相同的层在节点上**只存一份、只拉一次**。这是强制统一 base 镜像的真正理由。

In [ ]:
def node_storage_mb(n_services, base_mb, app_mb, shared_base=True):
    return base_mb + n_services * app_mb if shared_base else n_services * (base_mb + app_mb)

N, BASE, APP_MB = 20, 4000, 200
shared   = node_storage_mb(N, BASE, APP_MB, True)
separate = node_storage_mb(N, BASE, APP_MB, False)
print(f'{N} 个服务，共享底座: {shared/1000:>6.1f} GB')
print(f'{N} 个服务，各自底座: {separate/1000:>6.1f} GB')
print(f'倍数: {separate/shared:.1f}×')
assert separate / shared > 8, '共享底座应带来接近一个数量级的节省'
print('✅ 「所有服务 FROM 同一个内部 base」不是洁癖，是 10 倍的磁盘与带宽差异')

## 5 · 冷启动分解：镜像决策如何变成用户体验

$$T_{cold} = T_{sched} + \frac{S_{uncached}}{B} + T_{init} + \frac{W}{B_w} + T_{warmup}$$

这个数字直接约束模块 04 的自动扩缩策略——冷启动 90 秒的服务，反应式扩容必然来不及。

In [ ]:
def cold_start(image_mb_uncached, weights_gb, bandwidth_mbps=500, weight_bw_mbps=800,
               t_sched=3.0, t_init=6.0, t_warmup=15.0):
    t_pull = image_mb_uncached / bandwidth_mbps
    t_weights = weights_gb * 1024 / weight_bw_mbps
    total = t_sched + t_pull + t_init + t_weights + t_warmup
    return {'调度': t_sched, '拉镜像': t_pull, '进程初始化': t_init,
            '加载权重': t_weights, '预热': t_warmup, '合计': total}

configs = [
    ('单阶段镜像 + 权重打进镜像',              10412 + 14000, 0.0,  15.0),
    ('多阶段镜像 + 每次从对象存储拉权重',       3512,          14.0, 15.0),
    ('多阶段镜像 + 节点已缓存层 + 本地权重',    0,             0.5,  15.0),
    ('以上 + 编译/CUDA graph 缓存命中',        0,             0.5,   3.0),
]
for name, img_mb, w_gb, warm in configs:
    d = cold_start(img_mb, w_gb, t_warmup=warm)
    parts = ' + '.join(f'{k}{v:.0f}s' for k, v in d.items() if k != '合计')
    print(f'{name}\n  {parts} = **{d["合计"]:.0f}s**\n')

t_worst = cold_start(24412, 0.0)['合计']
t_best  = cold_start(0, 0.5, t_warmup=3.0)['合计']
assert t_worst > 5 * t_best, '最差与最优配置应差 5 倍以上'
print(f'✅ 最差 {t_worst:.0f}s vs 最优 {t_best:.0f}s —— 差 {t_worst/t_best:.1f} 倍。')
print('   这就是「扩容两分钟才缓解」和「十几秒就缓解」的区别，')
print('   也是模块 04 里「反应式扩容来不来得及」的分水岭。')

## ✏️ 练习 1：镜像实际体积

实现 `image_size_mb(layers)`：给定层列表（每层是 `{路径: 大小MB}`，值为 `-1` 表示白出标记），
返回**镜像实际体积**（所有非白出条目的大小之和，白出不减去下层字节）
和**可见体积**（overlay 后仍可见的文件大小之和）。返回 `(image_mb, visible_mb)`。

In [ ]:
def image_size_mb(layers):
    # TODO: image_mb = 所有层里非 -1 的值之和
    #       visible_mb = 从下往上 overlay（-1 表示删除该路径）后剩余文件大小之和
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
LS = [{'/base': 130.0}, {'/cache': 2000.0, '/lib': 50.0}, {'/cache': -1}, {'/app': 12.0}]
img, vis = image_size_mb(LS)
assert abs(img - 2192.0) < 1e-6, f'镜像应含被删除的 2000MB，得到 {img}'
assert abs(vis - 192.0) < 1e-6, f'可见应为 130+50+12=192，得到 {vis}'
# 同层内装了再删（根本没写进层）
img2, vis2 = image_size_mb([{'/base': 130.0}, {'/lib': 50.0}, {'/app': 12.0}])
assert abs(img2 - vis2) < 1e-6, '没有白出时，镜像体积应等于可见体积'
assert img2 < img
print(f'含白出: 镜像 {img}MB / 可见 {vis}MB   |   同层清理: 镜像 {img2}MB / 可见 {vis2}MB')
print('✅ 练习 1 通过：你复现了「分两条 RUN 删不掉字节」')

## ✏️ 练习 2：最优指令顺序

实现 `best_order(instrs, probs)`：在**保持依赖可行**的简化假设下（这里假设任意顺序都合法，
只有第 0 条 `FROM` 必须留在最前），返回使期望构建成本 `E = Σ p_i · Σ_{j≥i} c_j` 最小的顺序（下标列表）。

提示：这是一个经典的**调度排序**问题。对相邻两项交换做比较，可推出排序准则 ——
把 `p/c` 大的排在后面（即按 `p_i / c_i` **升序**排列，`FROM` 固定第一）。

In [ ]:
def best_order(instrs, probs):
    # TODO: 固定下标 0 在最前；其余按 probs[i]/instrs[i].cost_s 升序排列
    #       返回下标列表，例如 [0, 2, 1, 3]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
ins = [Instr('FROM','base',cost_s=5), Instr('COPY','. /app',cost_s=1),
       Instr('RUN','pip',cost_s=300), Instr('RUN','apt',cost_s=30)]
pr  = [0.01, 0.90, 0.05, 0.02]
order = best_order(ins, pr)
assert order[0] == 0, 'FROM 必须在最前'
assert sorted(order) == list(range(4)), '必须是一个排列'
assert order[-1] == 1, '天天改、又极便宜的 COPY . 应排在最后'

def E_of(order):
    return expected_build_cost([ins[i] for i in order], [pr[i] for i in order])
import itertools
brute = min(itertools.permutations(range(1,4)), key=lambda t: E_of((0,)+t))
assert abs(E_of(order) - E_of((0,)+brute)) < 1e-9, '应与暴力枚举的最优解一致'
print(f'最优顺序 {order}，期望成本 {E_of(order):.2f}s（暴力枚举同值 ✅）')
print('✅ 练习 2 通过：对拍暴力枚举，排序准则正确')

## ✏️ 练习 3：优雅停机的正确顺序

实现 `shutdown_sequence()`：返回优雅停机的**四个步骤的正确顺序**（字符串列表）。
可选步骤：`'mark_not_ready'`（就绪探针返回 503）、`'wait_endpoint_propagation'`（睡几秒等端点摘除生效）、
`'drain_inflight'`（等在途请求放完）、`'exit'`（退出进程）。

In [ ]:
def shutdown_sequence():
    # TODO: 返回 4 个步骤的正确顺序
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
seq = shutdown_sequence()
assert seq == ['mark_not_ready', 'wait_endpoint_propagation', 'drain_inflight', 'exit'], seq
assert seq.index('mark_not_ready') < seq.index('drain_inflight'), \
    '必须先摘流量再排空，否则排空期间还在进新请求'
assert seq.index('wait_endpoint_propagation') < seq.index('drain_inflight'), \
    '端点传播是最终一致的，必须等它生效（preStop: sleep 5）'
print('优雅停机顺序:', ' -> '.join(seq))
print('✅ 练习 3 通过：先摘流量、等传播、再排空、最后退出')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def image_size_mb(layers):
    image_mb = sum(v for lyr in layers for v in lyr.values() if v != -1)
    fs = {}
    for lyr in layers:
        for path, size in lyr.items():
            if size == -1: fs.pop(path, None)
            else:          fs[path] = size
    return image_mb, sum(fs.values())

In [ ]:
# 练习 2 参考答案
def best_order(instrs, probs):
    rest = sorted(range(1, len(instrs)), key=lambda i: probs[i] / instrs[i].cost_s)
    return [0] + rest

In [ ]:
# 练习 3 参考答案
def shutdown_sequence():
    return ['mark_not_ready', 'wait_endpoint_propagation', 'drain_inflight', 'exit']

---
## 🧪 真实数据胶囊：权重分发的隐形账单

扩容 20 个副本、每个从对象存储拉 140 GB 权重会发生什么？用公开的云计价量级算这笔账。
（带 try/except：本环境不联网，直接用内置的真实量级数字。）

In [ ]:
# 公开量级（约数，可改成你自己的报价）
WEIGHTS_GB      = 140.0     # Llama-70B fp16
EGRESS_USD_PER_GB = 0.09    # 跨区/出网数据传输典型价
SAME_REGION_USD_PER_GB = 0.01
OBJ_STORE_MBPS  = 800.0     # 单 Pod 从对象存储的有效拉取带宽

def scale_out_cost(n_replicas, gb, usd_per_gb, bw_mbps=OBJ_STORE_MBPS):
    total_gb = n_replicas * gb
    return {'总流量GB': total_gb,
            '流量费USD': total_gb * usd_per_gb,
            '每副本拉取秒': gb * 1024 / bw_mbps}

for n in [1, 5, 20]:
    a = scale_out_cost(n, WEIGHTS_GB, EGRESS_USD_PER_GB)
    print(f'{n:>3d} 副本跨区拉取: {a["总流量GB"]:>7.0f} GB, ${a["流量费USD"]:>7.2f}, '
          f'每副本等 {a["每副本拉取秒"]/60:.1f} 分钟')

cross = scale_out_cost(20, WEIGHTS_GB, EGRESS_USD_PER_GB)['流量费USD']
same  = scale_out_cost(20, WEIGHTS_GB, SAME_REGION_USD_PER_GB)['流量费USD']
print(f'\n跨区 ${cross:.0f}  vs  同区 ${same:.0f}  —— 每次扩容差 {cross/same:.0f} 倍')
assert cross > 200, '20 副本跨区拉 140GB 权重，单次扩容流量费超 $200'
print('✅ 这就是节点本地缓存 / 共享只读卷 / P2P 分发存在的根本原因')

**🧪 胶囊练习**：实现 `cache_savings(n_replicas, gb, usd_per_gb, replicas_per_node)`：
若采用**节点本地缓存**（同一节点上的多个副本共享一份已下载权重），只有**每个节点的第一个副本**产生下载。
返回 `(下载次数, 流量费USD)`。

In [ ]:
def cache_savings(n_replicas, gb, usd_per_gb, replicas_per_node):
    # TODO: 下载次数 = ceil(n_replicas / replicas_per_node)；流量费 = 下载次数 * gb * usd_per_gb
    raise NotImplementedError

In [ ]:
# 自测
downloads, cost = cache_savings(20, WEIGHTS_GB, EGRESS_USD_PER_GB, replicas_per_node=4)
assert downloads == 5, f'20 副本 / 每节点 4 个 = 5 次下载，得到 {downloads}'
assert abs(cost - 5 * 140 * 0.09) < 1e-6
naive = scale_out_cost(20, WEIGHTS_GB, EGRESS_USD_PER_GB)['流量费USD']
assert cost < naive / 3, '节点缓存应把流量费降到 1/4'
print(f'无缓存 ${naive:.0f} -> 节点缓存 ${cost:.0f}（下载 {downloads} 次而非 20 次）')
print('✅ 胶囊练习通过：节点本地缓存把扩容流量费降到 1/replicas_per_node')

In [ ]:
# 📖 胶囊参考答案
def cache_savings(n_replicas, gb, usd_per_gb, replicas_per_node):
    downloads = math.ceil(n_replicas / replicas_per_node)
    return downloads, downloads * gb * usd_per_gb

---
## 🔧 旁注：真实管线里这些对应什么

你在 Python 里验证过的逻辑，在真实系统里一一对应：

- **层与内容寻址** → OCI Image Spec 的 manifest + layer digest；`docker history` 看每层大小，`dive` 工具逐层浏览。
- **链式缓存失效** → BuildKit 的 DAG 求解器（比经典 builder 更聪明：能并行无依赖的阶段）。`docker build --progress=plain` 里的 `CACHED` 就是命中。
- **同层内装了再删** → `RUN pip install ... && rm -rf /root/.cache/pip`；现代做法是 `RUN --mount=type=cache,target=/root/.cache/pip`。
- **多阶段** → `FROM ... AS builder` + `COPY --from=builder`；CUDA 场景是 `devel` → `runtime` 基础镜像对。
- **权重分发** → initContainer + `aws s3 sync` / PVC(ReadOnlyMany) / DaemonSet 预热 hostPath / Dragonfly P2P。
- **优雅停机** → `lifecycle.preStop.exec: ["sleep","5"]` + 应用内 SIGTERM handler + `terminationGracePeriodSeconds: 120`（LLM 长生成必须调大）。

一份可直接用的生产 Dockerfile 骨架已在讲解第 4 节给出，建议在有 Docker 的机器上原样构建一遍，
对照 `docker history` 验证你在这里算出的层大小与顺序。

### 小结
- 镜像 = **一叠内容寻址的只读层** + config；overlay 合并、白出标记只影响可见性**不减体积**。
- 层缓存是**链式**的：一层失效、其后全废。「越少变的放越前」可由期望成本 `E = Σ p_i Σ_{j≥i} c_j` 证明。
- **多阶段构建**把编译工具链挡在最终镜像外，典型省 50%+；**共享 base** 让 20 个服务的节点存储省近 10 倍。
- **权重通常不进镜像**，但分离后必须把「模型版本」当一等公民管起来；扩容时的权重拉取有真实的、可能超过 GPU 本身的账单。
- **运行时契约六条**里，优雅停机对 LLM 最要命：先摘流量、等端点传播、再排空、最后退出。

下一站：**模块 02 · 推理服务 API** —— 镜像里的进程要对外说话了，契约怎么定、一个副本到底扛多少 QPS？